In [1]:
# ==========================================
# XGBoost Leak Detection Model Training
# Hardware-Matched, Real-Time Ready
# ==========================================

import pandas as pd
import numpy as np
import os
import glob
import random
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)
import joblib
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# CONFIGURATION
# ==========================================

DATA_PATH = '/kaggle/input/datasets/sarakhan24/trainingdata'
OUTPUT_DIR = '/kaggle/working'
os.makedirs(OUTPUT_DIR, exist_ok=True)

RANDOM_SEED = 42
WINDOW_SIZE = 30  # 30-second rolling window for normalization

# Train/Val/Test split (by scenarios)
TRAIN_SCENARIOS = 80
VAL_SCENARIOS = 24
TEST_SCENARIOS = 16
TOTAL_SCENARIOS = 120

# Feature columns (order matters for deployment!)
FEATURE_COLS = [
    'Flow_1_LPM_Norm',
    'Flow_2_LPM_Norm',
    'Flow_Div_Norm',
    'Flow_Div_Trend',
    'Pres_1_SPU_Norm',
    'Pres_2_SPU_Norm',
    'Pres_Div_Norm',
    'Pres_Div_Trend',
]

print("=" * 80)
print("XGBOOST LEAK DETECTION MODEL TRAINING")
print("Hardware-Matched, Real-Time Ready")
print("=" * 80)

# ==========================================
# STEP 1: LOAD DATA (FIXED FOR KAGGLE HEADERS)
# ==========================================

print("\n" + "=" * 80)
print("STEP 1: Loading Data from Kaggle Dataset")
print("=" * 80)

# Get all CSV files
csv_files = sorted(glob.glob(os.path.join(DATA_PATH, '*.csv')))
print(f"\nFound {len(csv_files)} scenario files")

if len(csv_files) == 0:
    raise RuntimeError(f"No CSV files found in {DATA_PATH}")

# Proper column names (matching your hardware data structure)
COLUMN_NAMES = ['Time_seconds', 'Pressure_1_SPU', 'Flow_1_LPM', 'Flow_2_LPM', 'Pressure_2_SPU', 'Label']

# Load all scenarios
scenarios = []
print(f"\nLoading scenarios:")

for i, filepath in enumerate(csv_files, 1):
    filename = os.path.basename(filepath)
    try:
        # Skip first row (Kaggle header), assign proper column names
        df = pd.read_csv(filepath, header=None, skiprows=1, names=COLUMN_NAMES)
        
        # Convert to numeric (handle any string artifacts)
        for col in COLUMN_NAMES:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        
        # Drop any NaN rows
        df = df.dropna()
        
        scenarios.append(df)
        if i % 20 == 0:
            print(f"  [{i:3d}/{len(csv_files)}] Loaded {filename}")
    except Exception as e:
        print(f"  ✗ Error loading {filename}: {e}")

print(f"  [{len(csv_files):3d}/{len(csv_files)}] Complete!")
print(f"\n✓ Successfully loaded {len(scenarios)} scenarios")

# Verify data structure
print(f"\nData structure (first file):")
print(f"  Rows: {len(scenarios[0])}")
print(f"  Columns: {list(scenarios[0].columns)}")
print(f"  First row:\n{scenarios[0].head(1).to_string()}")
print(f"  Data types:\n{scenarios[0].dtypes}")
# ==========================================
# STEP 2: FEATURE ENGINEERING
# ==========================================

print("\n" + "=" * 80)
print("STEP 2: Feature Engineering (Rolling Window Normalization)")
print("=" * 80)

def create_features_rolling(df, window_size=30): # Default to 30
    df = df.copy()
    # Flow Normalization
    for col in ['Flow_1_LPM', 'Flow_2_LPM']:
        r_min = df[col].rolling(window=window_size, min_periods=1).min()
        r_max = df[col].rolling(window=window_size, min_periods=1).max()
        df[f'{col}_Norm'] = (df[col] - r_min) / (r_max - r_min + 1e-6)
    
    # Flow Divergence
    df['Flow_Div'] = df['Flow_1_LPM'] - df['Flow_2_LPM']
    df['Flow_Div_Norm'] = (df['Flow_Div'] - df['Flow_Div'].rolling(window_size).min()) / (df['Flow_Div'].rolling(window_size).max() - df['Flow_Div'].rolling(window_size).min() + 1e-6)
    df['Flow_Div_Trend'] = df['Flow_Div'].rolling(window_size).mean()

    # Pressure Normalization
    for col in ['Pressure_1_SPU', 'Pressure_2_SPU']:
        r_min = df[col].rolling(window=window_size, min_periods=1).min()
        r_max = df[col].rolling(window=window_size, min_periods=1).max()
        df[f'{col.replace("Pressure_", "Pres_")}_Norm'] = (df[col] - r_min) / (r_max - r_min + 1e-6)

    # Pressure Divergence
    df['Pres_Div'] = df['Pressure_2_SPU'] - df['Pressure_1_SPU']
    df['Pres_Div_Norm'] = (df['Pres_Div'] - df['Pres_Div'].rolling(window_size).min()) / (df['Pres_Div'].rolling(window_size).max() - df['Pres_Div'].rolling(window_size).min() + 1e-6)
    df['Pres_Div_Trend'] = df['Pres_Div'].rolling(window_size).mean()
    
    return df.fillna(0)
# Process all scenarios
print(f"\nProcessing {len(scenarios)} scenarios with feature engineering...")

scenarios_with_features = []
for i, df in enumerate(scenarios, 1):
    df_features = create_features_rolling(df, window_size=WINDOW_SIZE)
    scenarios_with_features.append(df_features)
    
    if i % 20 == 0:
        print(f"  [{i:3d}/{len(scenarios)}] Processed")

print(f"  [{len(scenarios):3d}/{len(scenarios)}] Complete!")
print(f"\n✓ Feature engineering complete")
print(f"\nFeatures created:")
for feat in FEATURE_COLS:
    print(f"  - {feat}")

# ==========================================
# STEP 3: SHUFFLE AND SPLIT BY SCENARIOS
# ==========================================

print("\n" + "=" * 80)
print("STEP 3: Splitting Data (By Scenarios, Preserve Temporal Order)")
print("=" * 80)

# Dynamically adjust split based on actual number of scenarios
total_scenarios_loaded = len(scenarios_with_features)

print(f"\nTotal scenarios loaded: {total_scenarios_loaded}")

if total_scenarios_loaded < 50:
    print(f"⚠️  Warning: Only {total_scenarios_loaded} scenarios found (recommended: 120+)")
    print(f"   Adjusting split ratios...")
    
    # Adjust split for smaller dataset
    train_scenarios_count = max(int(total_scenarios_loaded * 0.65), 10)
    val_scenarios_count = max(int(total_scenarios_loaded * 0.20), 5)
    test_scenarios_count = total_scenarios_loaded - train_scenarios_count - val_scenarios_count
else:
    train_scenarios_count = 80
    val_scenarios_count = 24
    test_scenarios_count = 16

print(f"\nAdjusted split:")
print(f"  Train: {train_scenarios_count} scenarios")
print(f"  Val:   {val_scenarios_count} scenarios")
print(f"  Test:  {test_scenarios_count} scenarios")

# Shuffle scenarios (not individual rows)
random.seed(RANDOM_SEED)
indices = list(range(len(scenarios_with_features)))
random.shuffle(indices)

shuffled_scenarios = [scenarios_with_features[i] for i in indices]

# Split into train/val/test
train_scenarios = shuffled_scenarios[:train_scenarios_count]
val_scenarios = shuffled_scenarios[train_scenarios_count:train_scenarios_count + val_scenarios_count]
test_scenarios = shuffled_scenarios[train_scenarios_count + val_scenarios_count:]

# Safety check
if len(test_scenarios) == 0:
    print("\n⚠️  Warning: Not enough scenarios for test set!")
    print(f"   Moving 1 scenario from val to test...")
    test_scenarios = [val_scenarios.pop()]

# Concatenate
df_train = pd.concat(train_scenarios, ignore_index=True)
df_val = pd.concat(val_scenarios, ignore_index=True)
df_test = pd.concat(test_scenarios, ignore_index=True)

print(f"\nSplit configuration:")
print(f"  Train: {len(train_scenarios)} scenarios → {len(df_train):,} rows ({len(df_train)*100/len(pd.concat(scenarios_with_features)):.1f}%)")
print(f"  Val:   {len(val_scenarios)} scenarios → {len(df_val):,} rows ({len(df_val)*100/len(pd.concat(scenarios_with_features)):.1f}%)")
print(f"  Test:  {len(test_scenarios)} scenarios → {len(df_test):,} rows ({len(df_test)*100/len(pd.concat(scenarios_with_features)):.1f}%)")

# ========== CLASS DISTRIBUTION ==========

print(f"\nClass distribution:")

for split_name, df_split in [('Train', df_train), ('Val', df_val), ('Test', df_test)]:
    n_leak = (df_split['Label'] == 1).sum()
    n_no_leak = (df_split['Label'] == 0).sum()
    leak_pct = 100 * n_leak / len(df_split) if len(df_split) > 0 else 0
    print(f"  {split_name}:")
    print(f"    No-Leak: {n_no_leak:,} ({100-leak_pct:.1f}%)")
    print(f"    Leak:    {n_leak:,} ({leak_pct:.1f}%)")

# ========== EXTRACT FEATURES AND LABELS ==========

X_train = df_train[FEATURE_COLS].values
y_train = df_train['Label'].values

X_val = df_val[FEATURE_COLS].values
y_val = df_val['Label'].values

X_test = df_test[FEATURE_COLS].values
y_test = df_test['Label'].values

print(f"\n✓ Data split complete")
# ==========================================
# STEP 4: HANDLE CLASS IMBALANCE
# ==========================================

print("\n" + "=" * 80)
print("STEP 4: Class Imbalance Configuration")
print("=" * 80)

n_no_leak_train = (y_train == 0).sum()
n_leak_train = (y_train == 1).sum()
scale_pos_weight = n_no_leak_train / n_leak_train if n_leak_train > 0 else 1.0

print(f"\nClass imbalance ratio (no-leak : leak): {scale_pos_weight:.2f}:1")
print(f"XGBoost will penalize false negatives {scale_pos_weight:.2f}x more")
print(f"→ Prioritizes catching leaks over perfect accuracy")

# ==========================================
# STEP 5: TRAIN XGBOOST
# ==========================================

print("\n" + "=" * 80)
print("STEP 5: Training XGBoost")
print("=" * 80)

xgb_params = {
    'n_estimators': 150,  # Reduced from 200 since no early stopping
    'max_depth': 5,
    'learning_rate': 0.1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'scale_pos_weight': scale_pos_weight,
    'eval_metric': 'logloss',
    'random_state': RANDOM_SEED,
    'n_jobs': -1,
    'tree_method': 'hist',
}

print(f"\nXGBoost parameters:")
for key, value in xgb_params.items():
    print(f"  {key}: {value}")

# Create and train model
clf = XGBClassifier(**xgb_params)

print(f"\nTraining on {len(X_train):,} rows...")

clf.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    verbose=100
)

print(f"\n✓ Training complete")
# ==========================================
# STEP 6: EVALUATION
# ==========================================

print("\n" + "=" * 80)
print("STEP 6: Model Evaluation")
print("=" * 80)

def evaluate_model(clf, X, y, set_name="Set"):
    """Evaluate model on a dataset"""
    y_pred = clf.predict(X)
    y_pred_proba = clf.predict_proba(X)[:, 1]
    
    acc = accuracy_score(y, y_pred)
    prec = precision_score(y, y_pred, zero_division=0)
    rec = recall_score(y, y_pred, zero_division=0)
    f1 = f1_score(y, y_pred, zero_division=0)
    auc = roc_auc_score(y, y_pred_proba)
    
    print(f"\n{set_name} Metrics:")
    print(f"  Accuracy:  {acc:.4f}")
    print(f"  Precision: {prec:.4f} (avoid false positives!)")
    print(f"  Recall:    {rec:.4f} (catch all leaks!)")
    print(f"  F1-Score:  {f1:.4f}")
    print(f"  ROC-AUC:   {auc:.4f}")
    
    return y_pred, y_pred_proba, {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1, 'auc': auc}

# Evaluate all sets
y_pred_train, y_pred_proba_train, metrics_train = evaluate_model(clf, X_train, y_train, "Training Set")
y_pred_val, y_pred_proba_val, metrics_val = evaluate_model(clf, X_val, y_val, "Validation Set")
y_pred_test, y_pred_proba_test, metrics_test = evaluate_model(clf, X_test, y_test, "Test Set")

# ========== CONFUSION MATRICES ==========

print("\n" + "-" * 80)
print("Confusion Matrices")
print("-" * 80)

def plot_confusion_matrix(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    
    plt.figure(figsize=(7, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['No Leak', 'Leak'],
                yticklabels=['No Leak', 'Leak'],
                annot_kws={'size': 14})
    plt.xlabel('Predicted', fontsize=12)
    plt.ylabel('Actual', fontsize=12)
    plt.title(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f'confusion_matrix_{title.lower().replace(" ", "_")}.png'), dpi=100)
    plt.close()
    
    # Print interpretation
    tn, fp, fn, tp = cm.ravel()
    print(f"\n{title}:")
    print(f"  True Negatives:  {tn:,} (correct no-leak predictions)")
    print(f"  False Positives: {fp:,} (⚠️ false alarms - MINIMIZE!)")
    print(f"  False Negatives: {fn:,} (⚠️ missed leaks)")
    print(f"  True Positives:  {tp:,} (correct leak detections)")

plot_confusion_matrix(y_train, y_pred_train, "Training Set")
plot_confusion_matrix(y_val, y_pred_val, "Validation Set")
plot_confusion_matrix(y_test, y_pred_test, "Test Set")

# ========== DETAILED CLASSIFICATION REPORT ==========

print("\n" + "-" * 80)
print("Detailed Classification Report (Test Set)")
print("-" * 80)

print(classification_report(y_test, y_pred_test, target_names=['No Leak', 'Leak']))

# ========== FEATURE IMPORTANCE ==========

print("\n" + "-" * 80)
print("Feature Importance")
print("-" * 80)

importance_df = pd.DataFrame({
    'Feature': FEATURE_COLS,
    'Importance': clf.feature_importances_
}).sort_values('Importance', ascending=False)

print(importance_df.to_string(index=False))

# Plot feature importance
plt.figure(figsize=(10, 6))
sns.barplot(data=importance_df, x='Importance', y='Feature', hue='Feature', legend=False, palette='viridis')
plt.title('Feature Importance for Leak Detection', fontsize=14, fontweight='bold')
plt.xlabel('Importance Score', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'feature_importance.png'), dpi=100)
plt.close()

print(f"\n✓ Feature importance plot saved")

# ==========================================
# STEP 7: SAVE MODEL AND DEPLOYMENT CONFIG
# ==========================================

print("\n" + "=" * 80)
print("STEP 7: Saving Model and Deployment Configuration")
print("=" * 80)

# Save model
model_path = os.path.join(OUTPUT_DIR, 'xgb_leak_detector_hardware.json')
clf.save_model(model_path)
print(f"\n✓ Model saved: {model_path}")

# Save deployment configuration
config = {
    'model_path': 'xgb_leak_detector_hardware.json',
    'window_size': WINDOW_SIZE,
    'feature_columns': FEATURE_COLS,
    'n_features': len(FEATURE_COLS),
    'input_columns': ['Time_seconds', 'Pressure_1_SPU', 'Flow_1_LPM', 'Flow_2_LPM', 'Pressure_2_SPU'],
    'training_config': {
        'total_scenarios': len(scenarios_with_features),
        'train_scenarios': len(train_scenarios),
        'val_scenarios': len(val_scenarios),
        'test_scenarios': len(test_scenarios),
        'total_rows': len(pd.concat(scenarios_with_features)),
    },
    'metrics': {
        'train': metrics_train,
        'val': metrics_val,
        'test': metrics_test,
    }
}

config_path = os.path.join(OUTPUT_DIR, 'deployment_config.json')
import json
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

print(f"✓ Deployment config saved: {config_path}")

# Save feature engineering code as Python file for easy import
deployment_code = '''
"""
Deployment Feature Engineering for Leak Detection
Use this on hardware/real-time system
"""

import pandas as pd
import numpy as np

WINDOW_SIZE = 60

FEATURE_COLS = [
    'Flow_1_LPM_Norm',
    'Flow_2_LPM_Norm',
    'Flow_Div_Norm',
    'Flow_Div_Trend',
    'Pres_1_SPU_Norm',
    'Pres_2_SPU_Norm',
    'Pres_Div_Norm',
    'Pres_Div_Trend',
]

def create_features_rolling(df, window_size=WINDOW_SIZE):
    """
    Create features using rolling window normalization.
    
    Safe for real-time deployment:
    - Uses only historical data (past window_size samples)
    - No future data leakage
    - Adapts to hardware drift automatically
    
    Args:
        df: DataFrame with columns [Time_seconds, Pressure_1_SPU, Flow_1_LPM, Flow_2_LPM, Pressure_2_SPU, Label]
        window_size: Size of rolling window (60 seconds recommended)
    
    Returns:
        df: DataFrame with engineered features
    """
    
    df = df.copy()
    
    # Normalize flows
    for col in ['Flow_1_LPM', 'Flow_2_LPM']:
        rolling_min = df[col].rolling(window=window_size, min_periods=1).min()
        rolling_max = df[col].rolling(window=window_size, min_periods=1).max()
        df[f'{col}_Norm'] = (df[col] - rolling_min) / (rolling_max - rolling_min + 1e-6)
    
    # Flow divergence
    df['Flow_Div'] = df['Flow_1_LPM'] - df['Flow_2_LPM']
    rolling_min_div = df['Flow_Div'].rolling(window=window_size, min_periods=1).min()
    rolling_max_div = df['Flow_Div'].rolling(window=window_size, min_periods=1).max()
    df['Flow_Div_Norm'] = (df['Flow_Div'] - rolling_min_div) / (rolling_max_div - rolling_min_div + 1e-6)
    df['Flow_Div_Trend'] = df['Flow_Div'].rolling(window=window_size, min_periods=1).mean()
    
    # Normalize pressures
    for col in ['Pressure_1_SPU', 'Pressure_2_SPU']:
        rolling_min = df[col].rolling(window=window_size, min_periods=1).min()
        rolling_max = df[col].rolling(window=window_size, min_periods=1).max()
        df[f'{col.replace("Pressure_", "Pres_")}_Norm'] = (
            (df[col] - rolling_min) / (rolling_max - rolling_min + 1e-6)
        )
    
    # Pressure divergence
    df['Pres_Div'] = df['Pressure_2_SPU'] - df['Pressure_1_SPU']
    rolling_min_pres = df['Pres_Div'].rolling(window=window_size, min_periods=1).min()
    rolling_max_pres = df['Pres_Div'].rolling(window=window_size, min_periods=1).max()
    df['Pres_Div_Norm'] = (df['Pres_Div'] - rolling_min_pres) / (rolling_max_pres - rolling_min_pres + 1e-6)
    df['Pres_Div_Trend'] = df['Pres_Div'].rolling(window=window_size, min_periods=1).mean()
    
    df = df.fillna(0)
    
    return df

def prepare_for_inference(df):
    """Prepare raw data for model inference"""
    df = create_features_rolling(df, window_size=WINDOW_SIZE)
    return df[FEATURE_COLS].values

# Example usage:
# df_raw = pd.read_csv('sensor_data.csv')
# features = prepare_for_inference(df_raw)
# predictions = model.predict(features)
'''

deployment_code_path = os.path.join(OUTPUT_DIR, 'deployment_features.py')
with open(deployment_code_path, 'w') as f:
    f.write(deployment_code)

print(f"✓ Deployment code saved: {deployment_code_path}")

# ==========================================
# STEP 8: SUMMARY AND RECOMMENDATIONS
# ==========================================

print("\n" + "=" * 80)
print("TRAINING SUMMARY")
print("=" * 80)

print(f"\n📊 Dataset:")
print(f"  Total scenarios: {len(scenarios_with_features)}")
print(f"  Total rows: {len(pd.concat(scenarios_with_features)):,}")
print(f"  Window size: {WINDOW_SIZE} seconds")

print(f"\n🎯 Model Performance:")
print(f"  Test Accuracy:  {metrics_test['accuracy']:.4f}")
print(f"  Test Precision: {metrics_test['precision']:.4f} ← IMPORTANT (false positives)")
print(f"  Test Recall:    {metrics_test['recall']:.4f} ← IMPORTANT (catch leaks)")
print(f"  Test F1-Score:  {metrics_test['f1']:.4f}")
print(f"  Test ROC-AUC:   {metrics_test['auc']:.4f}")

print(f"\n💾 Saved Files:")
print(f"  Model: {model_path}")
print(f"  Config: {config_path}")
print(f"  Deployment Code: {deployment_code_path}")
print(f"  Visualizations: confusion_matrix_*.png, feature_importance.png")

print(f"\n⚠️ Critical Metrics for Deployment:")
print(f"  False Positive Rate: {metrics_test['precision']:.1%} precision = {100-metrics_test['precision']*100:.1f}% false alarm rate")
print(f"  False Negative Rate: {metrics_test['recall']:.1%} recall = {100-metrics_test['recall']*100:.1f}% miss rate")

if metrics_test['precision'] > 0.95:
    print(f"  ✅ High precision: Safe for deployment (low false alarms)")
else:
    print(f"  ⚠️ Lower precision: May have false alarms, monitor closely")

if metrics_test['recall'] > 0.90:
    print(f"  ✅ High recall: Catches most leaks")
else:
    print(f"  ⚠️ Lower recall: May miss some leaks")

print(f"\n🚀 Deployment Instructions:")
print(f"  1. Load model: clf = xgb.XGBClassifier()")
print(f"     clf.load_model('xgb_leak_detector_hardware.json')")
print(f"  2. Import features: from deployment_features import prepare_for_inference")
print(f"  3. For each new sensor reading:")
print(f"     - Add to buffer (last 60 seconds)")
print(f"     - features = prepare_for_inference(df_buffer)")
print(f"     - prediction = clf.predict(features[-1])")
print(f"     - if prediction == 1: RAISE_ALARM()")

print(f"\n" + "=" * 80)
print("✓ TRAINING COMPLETE")
print("=" * 80)

XGBOOST LEAK DETECTION MODEL TRAINING
Hardware-Matched, Real-Time Ready

STEP 1: Loading Data from Kaggle Dataset

Found 87 scenario files

Loading scenarios:
  [ 20/87] Loaded leak_large_k1.5e-05_sudden_full_0.10noise_083.csv
  [ 40/87] Loaded leak_large_k7.8e-06_sudden_full_0.05noise_061.csv
  [ 60/87] Loaded leak_micro_k1.5e-06_spike_0.15noise_024.csv
  [ 80/87] Loaded leak_small_k4.0e-06_spike_0.10noise_044.csv
  [ 87/87] Complete!

✓ Successfully loaded 87 scenarios

Data structure (first file):
  Rows: 299
  Columns: ['Time_seconds', 'Pressure_1_SPU', 'Flow_1_LPM', 'Flow_2_LPM', 'Pressure_2_SPU', 'Label']
  First row:
   Time_seconds  Pressure_1_SPU  Flow_1_LPM  Flow_2_LPM  Pressure_2_SPU  Label
0             1           -0.86         3.0        2.65          -0.096      0
  Data types:
Time_seconds        int64
Pressure_1_SPU    float64
Flow_1_LPM        float64
Flow_2_LPM        float64
Pressure_2_SPU    float64
Label               int64
dtype: object

STEP 2: Feature Engineeri